<a href="https://colab.research.google.com/github/gitmystuff/DTSC3010/blob/main/Week_11/Data_Science_Fiction_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Science Fiction II

Your Name

## Getting Started

* Colab - get notebook from gitmystuff DTSC3010 repository
* Save a Copy in Drive
* Remove Copy of
* Edit name
* Clean up Colab Notebooks folder
* Submit shared link

## Instructions

The goal of this assignment is to take messy data, clean it up, and then analyze it using logistic regression.

* Run Parts 1 and 2 being careful to take in what's going on
* In Part 3 you are asked to clean the data in preparation for modeling
* Part 4 - perform necessary feature engineering
* Part 5 - select the variables that will be more useful for classification
* Part 6 - model the data and evaluate, explain concepts when asked

# Part 1 - The Data

## Seed the Project

In [ ]:
import time
import numpy as np
import random

def generate_user_seed():
    # Get current time in nanoseconds (more granular)
    nanoseconds = time.time_ns()

    # Add a small random component to further reduce collision chances
    random_component = random.randint(0, 1000)  # Adjust range as needed

    # Combine them (XOR is a good way to mix values)
    seed = nanoseconds ^ random_component

    # Ensure the seed is within the valid range for numpy's seed
    seed = seed % (2**32)  # Modulo to keep it within 32-bit range

    return seed

user_seed = generate_user_seed()
print(user_seed)
random_state = np.random.seed(user_seed)

## Faker

In [ ]:
pip install Faker -q

In [ ]:
# this could have been a list of 50 states, 50 counties, countries, zipcodes, etc.
habitable_planets = [
    "Alpha Centauri III",
    "Eden",
    "Terra Nova",
    "Tiberius",
    "Vega Colony",
    "Cait",
    "Andoria",
    "Vulcanis",
    "Risa",
    "Betazed",
    "Ba'ku",
    "Aldea",
    "Nimbus III",
    "Deneva",
    "Capella IV",
    "Organia",
    "Trillius Prime",
    "Kaelon II",
    "Mintaka III",
    "Rubicun III",
    "Pacifica",
    "Tau Ceti III",
    "Melina",
    "Argelius II",
    "Iconia",
    "Alderaan",
    "Naboo",
    "Bespin (Cloud City)",
    "Yavin IV",
    "Endor (Forest Moon)",
    "Kashyyyk",
    "Mon Cala",
    "Corellia",
    "Chandrila",
    "Ryloth",
    "Cato Neimoidia",
    "Felucia",
    "Saleucami",
    "Stewjon",
    "Iego",
    "Glee Anselm",
    "Mirial",
    "Serenno",
    "Malastare",
    "Dantooine",
    "Haruun Kal",
    "Manaan",
    "Zolan",
    "Ord Mantell",
    "Pantora"
]

In [ ]:
# create demographic data
import numpy as np
import pandas as pd
from faker import Faker
fake = Faker()

n = 1000

output = []
for x in range(n):
    biology = np.random.choice(['Cytophore', 'Kymete'], p=[0.5, 0.5])
    output.append({
        'categorical_1': biology,
        'categorical_2': np.random.choice(['Xylosian', 'Veridian', 'CKaeltharr']),
        'name_1': fake.first_name_female() if biology == 'Cytophore' else fake.first_name_male(),
        'name_2': fake.last_name(),
        'code': fake.zipcode(),
        'date': fake.date_of_birth(),
        'location': np.random.choice(habitable_planets)
    })

demographics = pd.DataFrame(output)
print(demographics.shape)
demographics.head()

## Create Independent Variable Correlated with Class

In [ ]:
import numpy as np
import pandas as pd

def generate_feature(df, class_col, coeff, intercept):
    """
    Generates normally distributed feature data for a logistic regression model.

    Args:
        df: The pandas DataFrame containing the class column.
        class_col: The name of the class column (containing 0s and 1s).
        coeff: The coefficient for the feature in the logistic regression model.
        intercept: The intercept of the logistic regression model.

    Returns:
        A pandas Series containing the generated feature data.
    """

    # Generate probabilities based on the class
    probs = np.random.rand(len(df))  # Initial random probabilities
    probs = np.where(df[class_col] == 1, probs * 0.8 + 0.2, probs * 0.8)  # Adjust for class

    # Apply the inverse logit (logit) function
    logits = np.log(probs / (1 - probs))

    # Calculate the feature values
    feature_values = (logits - intercept) / coeff

    return pd.Series(feature_values)



## Make Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# 1. Generate continuous regression data instead of classification data
# 'noise' adds variance around the underlying linear model
X, y = make_regression(n_samples=n, n_features=2, n_informative=2, noise=15.0, random_state=42)

df = pd.DataFrame(X, columns=['informative_1', 'informative_2'])

# Assuming 'demographics' is defined earlier in your notebook
df = pd.concat([demographics, df], axis=1).reset_index(drop=True)

# 2. Fit the Linear Regression model ONCE (outside the apply function)
model = LinearRegression()
model.fit(X, y)
coefficients = model.coef_
intercept = model.intercept_

def make_linear_y(row):
    # Note: LinearRegression coef_ is a 1D array for a single target,
    # so we use coefficients[0] and coefficients[1] instead of coefficients[0][0]
    f_of_x = intercept + (coefficients[0] * row['informative_1']) + (coefficients[1] * row['informative_2'])
    return f_of_x

# 3. Assign the variables
# 'target_perfect' is the exact linear plane without the noise (the independent feature)
df['target_perfect'] = df.apply(make_linear_y, axis=1)

# 'target' is the actual dependent variable with noise included
df['target'] = y

# 4. Generate the correlated feature (assuming generate_feature supports continuous targets)
df['corr_feature_target'] = generate_feature(df, 'target', 0.5, -1)

df.head()

## Automation Functions

1. gen_null(series, perc)
2. gen_quasi_constants(primary_label, variation_percentage=.2, replace=False)
3. gen_normal_data(mu=0, std=1, size=len(df))
4. gen_uniform_data(size=len(df))
5. gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df))
6. gen_correlated_normal_series(original_series, target_correlation, size=len(df))
7. gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df))
8. gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3)
9. gen_standard_scaling(mean=50, std_dev=10, size=len(df), scale_factor=1000)
10. gen_minmax_scaling(mean=50, std_dev=10, size=len(df), range_factor=10)
11. random_choice_data(choices, size)

In [ ]:
# functions
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize


def gen_null(series, perc):
  """
  Introduces null values (np.nan) into a list based on a specified percentage.

  Args:
      var: The variable to modify.
      perc: The percentage of values to replace with nulls (0-100).

  Returns:
      The modified variable with null.
  """
  var = series.copy()
  num_nulls = int(len(var) * (perc / 100))
  indices_to_replace = np.random.choice(len(var), num_nulls, replace=False)

  for idx in indices_to_replace:
      var[idx] = np.nan

  return var

def gen_quasi_constants(primary_label, variation_percentage=.2, size=len(df)):
  """
  Generates quasi-constant labels for a Series, with a small percentage of variation.

  Args:
      primary_label: The main label to use for most values.
      variation_percentage: The percentage of labels to vary (0-100).

  Returns:
      A new Series containing the quasi-constant labels.
  """

  series = pd.Series(np.full(size, primary_label))
  num_variations = int(size * (variation_percentage / 100))
  variation_indices = np.random.choice(series.index, num_variations, replace=False)
  primary_label = primary_label + '_0'
  variation1 = primary_label + '_1'
  variation2 = primary_label + '_2'

  labels = pd.Series([primary_label] * len(series), index=series.index)
  labels.loc[variation_indices] = np.random.choice([variation1, variation2], size=num_variations)  # Adjust variations as needed

  return labels

def gen_normal_data(mu=0, std=1, size=len(df)):
  """
  Generates a normal dataset given the mean and standard deviation

  Args:
        mu: The mean of the normal distribution.
        std: The standard deviation of the normal distribution.
        size: The number of data points to generate.

  Returns:
        A normally distributed series.
  """
  return np.random.normal(mu, std, size)

def gen_uniform_data(size=len(df)):
  """
  Generates a uniform dataset

  Args:
        size: The number of data points to generate.

  Returns:
        A uniform distributed series.
  """
  return np.random.uniform(size=size)

def gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df)):
  """
  Generates two datasets with a multivariate normal distribution given the mean and covariance matrix

  Args:
        mean: The mean of each of the datasets.
        cov: The covariance matrix of the datasets.
        size: The number of data points to generate.

  Returns:
        Two correlated series.
  """
  ds1, ds2 = np.random.multivariate_normal(mean, cov, size, tol=1e-6).T # ds = dataset
  return ds1, ds2

def gen_correlated_normal_series(original_series, target_correlation, size=len(df)):
  """
  Generates a correlated series based on a given series.

  This function takes an original series as input and generates a new series
  that is correlated with the original series. The correlation between the
  original and generated series is approximately equal to the specified
  target correlation.

  The generated series is created by linearly transforming the original series
  and adding Gaussian noise with an adjusted standard deviation to achieve the
  desired correlation.

  Args:
      original_series (numpy.ndarray): The original series.
      target_correlation (float): The desired Pearson correlation coefficient
          between the original and generated series.

  Returns:
      numpy.ndarray: The generated correlated series.
  """
  return np.mean(original_series) + target_correlation * (original_series - np.mean(original_series)) \
  +  np.random.normal(0, np.sqrt(1 - target_correlation**2) * np.std(original_series), len(original_series))
  """
  Explanation

  This one-liner leverages the properties of linear transformations and normal distributions to generate a correlated series.

  It first centers the original_series by subtracting its mean.
  It then scales this centered series by the target_correlation.
  Finally, it adds Gaussian noise with a standard deviation adjusted to ensure the overall correlation matches the target_correlation.
  """

def gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df)):
  """
  Work in progress

  Generates a new series correlated with the given series based on the specified correlation coefficient,
  using rank correlation to ensure the generated series follows a uniform distribution.

  Args:
      original_series (numpy.ndarray or list): The original series.
      correlation_coefficient (float): The desired correlation coefficient between the original and generated series.
      size: The number of data points to generate.

  Returns:
      The generated correlated series with a uniform distribution.
  """
  z_scores = (original_series - np.mean(original_series)) / np.std(original_series)
  correlation_coefficient=.7
  return norm.cdf(correlation_coefficient * norm.ppf(np.random.uniform(size=size)) + np.sqrt(1 - correlation_coefficient**2) * z_scores)

def pearson_r_func(x, y, y_mean, y_std, desired_r):
    x_mean = np.mean(x)
    x_std = np.std(x)
    numerator = np.sum((x - x_mean) * (y - y_mean))
    denominator = x_std * y_std * len(x)
    calculated_r = numerator / denominator
    return (calculated_r - desired_r)**2  # Minimize the squared difference

def minimize_r(original_series, target_correlation, size=len(df)):
    y = original_series
    y_mean = np.mean(y)
    y_std = np.std(y)
    desired_r = target_correlation

    # Initial guess for x values
    x0 = np.random.uniform(size=len(original_series))

    # Solve for x
    result = minimize(pearson_r_func, x0, args=(y, y_mean, y_std, desired_r))

    if result.success:
        x_solution = result.x
        # print("Solution for x:", x_solution)
        return x_solution
    else:
        print("Optimization failed.")

def gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3):
    """
    Generates a normal distribution with outliers.

    Args:
        mean (float): The mean of the normal distribution.
        std_dev (float): The standard deviation of the normal distribution.
        size (int): The number of samples to generate.
        outlier_percentage (float): The percentage of outliers to introduce (between 0 and 1).
        outlier_magnitude (float): The magnitude by which outliers deviate from the mean.

    Returns:
        numpy.ndarray: The generated data with outliers.
    """
    data = np.random.normal(mean, std_dev, size)
    num_outliers = int(size * outlier_percentage)
    outlier_indices = np.random.choice(size, num_outliers, replace=False)
    for index in outlier_indices:
        if np.random.rand() < 0.5:
            data[index] += outlier_magnitude
        else:
            data[index] -= outlier_magnitude

    return data

def gen_standard_scaling(mean=50, std_dev=10, size=len(df), scale_factor=1000):
  """
  Generates data with a specified mean and standard deviation, then scales it by a factor to create a distribution needing scaling.

  Args:
      mean (float): The mean of the original distribution.
      std_dev (float): The standard deviation of the original distribution.
      size (int): The number of samples to generate.
      scale_factor (float): The factor by which to scale the original distribution.

  Returns:
      numpy.ndarray: The generated data needing scaling.
  """
  original_data = np.random.normal(mean, std_dev, size)
  return original_data * scale_factor

def gen_minmax_scaling(mean=50, std_dev=10, size=len(df), range_factor=10):
  """
  Generates data with a specified mean and standard deviation, then scales and shifts it to create a distribution needing MinMax scaling.

  Args:
      mean (float): The mean of the original distribution.
      std_dev (float): The standard deviation of the original distribution.
      size (int): The number of samples to generate.
      range_factor (float): The factor to expand the range of the original distribution.

  Returns:
      numpy.ndarray: The generated data needing scaling.
  """

  # Generate the original data
  original_data = np.random.normal(mean, std_dev, size)

  # Expand the range of the data
  min_val = np.min(original_data)
  max_val = np.max(original_data)
  return (original_data - min_val) * range_factor + min_val

def random_choice_data(choices, size):
  """
  Generates a new series correlated with the given series based on the specified correlation coefficient,
  using rank correlation to ensure the generated series follows a uniform distribution.

  Args:
      original_series (numpy.ndarray or list): The original series.
      correlation_coefficient (float): The desired correlation coefficient between the original and generated series.

  Returns:
      numpy.ndarray: The generated correlated series with a uniform distribution.
  """
  return np.random.choice(choices, size=size)


In [ ]:
# categorical variables with little correlation to target
df['random choice 2'] = random_choice_data(['Rand Choice 1', 'Rand Choice 2'], size=len(df))
df['random choice 4'] = random_choice_data(['North', 'South', 'East', 'West'], size=len(df))
df['random choice 7'] = random_choice_data(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], size=len(df))

# categorical random choices with random # of labels
num_labels = np.random.randint(3, 5)
df[f'random label num {num_labels}'] = random_choice_data([f'label num lo {i}' for i in range(1, num_labels + 1)], size=len(df))

num_labels = np.random.randint(10, 15)
df[f'random label num {num_labels}'] = random_choice_data([f'label num hi {i}' for i in range(1, num_labels + 1)], size=len(df))

In [ ]:
# categorical variables correlated with target
df['pd qcut1'] = pd.qcut(df['target'], 2, labels=['Low', 'High']) # bi label
df['pd qcut2'] = pd.qcut(df['target'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4']) # 4 labels

quantiles = [0, 0.1, 0.2, 0.4, 0.6, 0.8, 1]
df['pd qcut3'] = pd.qcut(df['target'], quantiles, labels=['G1', 'G2', 'G3', 'G4', 'G5', 'G6']) # 6 labels

In [ ]:
# generate four numerical normally distributed continuous features that have a correlation greater than absolute value of .5 with each other
# gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df))
df['multicollinearity 1'], df['multicollinearity 2'] = gen_multivariate_normal_data(mean=[0, 0], cov=[[1, .7], [.7, 1]], size=len(df))
df['multicollinearity 3'], df['multicollinearity 4'] = gen_multivariate_normal_data(mean=[0, 0], cov=[[1, .9], [.9, 1]], size=len(df))

In [ ]:
# generate two normally distributed features that are correlated with the target
# gen_correlated_normal_series(original_series, target_correlation, size=len(df))
df['correlated w target 1'] = gen_correlated_normal_series(df['target'], target_correlation=.5)
df['correlated w target 2'] = gen_correlated_normal_series(df['target'], target_correlation=.7)
df.info()

In [ ]:
# generate two uniformly distributed features that are correlated with the target
# gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df))
df['uniform corr 1'] = gen_correlated_uniform_series(df['target'])
df['uniform corr 2'] = gen_correlated_uniform_series(df['target'])

In [ ]:
# create two features that are duplicates of other features
df['duplicate_1'] = df['informative_1']
df['duplicate_2'] = df['informative_2']

In [ ]:
# create two numerical features with outliers
df['outliers 1'] = gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3)
df['outliers 2'] = gen_outliers(mean=3, std_dev=2, size=len(df), outlier_percentage=0.2, outlier_magnitude=2)

In [ ]:
# create a numerical feature that needs standard scaling
df['standard scaling'] = gen_standard_scaling()

In [ ]:
# create a numerical feature that needs min max scaling
df['min max scaling'] = gen_minmax_scaling()

In [ ]:
# generate null values
for col in df.drop(['target', 'informative_1', 'informative_2', 'target_perfect', 'duplicate_1', 'duplicate_2'], axis=1).columns:
    df[col] = gen_null(df[col], np.random.choice([0, 5, 10, 20, 30, 50], size=1).item())

In [ ]:
# create two features that have constant values
df['constant_1'] = 'constant_value'
df['constant_2'] = 'constant_value'

In [ ]:
# create two features with semi constant values
df['semi_constant_1'] = gen_quasi_constants('q_const', variation_percentage = 1)
df['semi_constant_2'] = gen_quasi_constants('q_const', variation_percentage = 1)

In [ ]:
print(df.info())  # check progress

In [ ]:
# add duplicates (still great for a cleaning exercise!)
dupes = df.loc[0:9]
df = pd.concat([df, dupes], axis=0).reset_index(drop=True)

# shuffle selected columns
demographic_columns = demographics.columns
remaining_columns = [col for col in df.columns if col not in demographic_columns]
np.random.shuffle(remaining_columns)

# Reassemble the DataFrame with the shuffled columns
df = df[list(demographic_columns) + list(remaining_columns)]

# move 'target' to the end of the list (updated from 'class')
target_var = 'target'
df = df[df.drop(target_var, axis=1).columns.tolist() + [target_var]]

print(df.shape)
print(df.info())
df.head()

In [ ]:
df.to_csv('data science fiction ii pt 1.csv', index=False)

# Part 2 - Exploratory Data Analysis (EDA)

Exploratory data analysis (EDA) is a data analysis method that helps data scientists understand their data and identify patterns. It's often used as the first step in data analysis.

## Load Data

In [ ]:
import pandas as pd

df = pd.read_csv('data science fiction ii pt 1.csv')
print(df.shape)
print(df.info())
df.head()

## Var Types

Identifying variable types is the "measure twice, cut once" phase of data science. Before you can build a model or even create a simple plot, you have to know what kind of data you’re holding.

Here is why this step is foundational:

### 1. Choosing the Right Statistical Strategy
Statistical tests are picky. You cannot perform the same math on a zip code (categorical) as you would on a salary (numerical).
* **Numerical Data:** You can calculate the mean, variance, and standard deviation.
* **Categorical Data:** These operations are meaningless. Instead, you look at mode, frequency, and proportions.

### 2. Directing Exploratory Data Analysis (EDA)
Your variable types dictate your visualizations. If you pick the wrong chart for your data type, you’ll likely end up with a confusing mess rather than an insight.
* **Continuous variables** (like temperature) call for histograms or box plots to see distribution.
* **Discrete/Categorical variables** (like car brand) call for bar charts or pie charts to see counts.



### 3. Requirements for Machine Learning
Most machine learning algorithms are essentially giant calculators—they only understand numbers.
* **Encoding:** If you have categorical data (e.g., "Red," "Blue," "Green"), you must identify them so you can apply techniques like **One-Hot Encoding** or **Label Encoding** to turn them into a format the model can process.
* **Feature Scaling:** Certain models (like K-Nearest Neighbors or SVMs) are sensitive to the scale of numerical data. You need to identify these variables to apply normalization or standardization.

### 4. Data Cleaning and Error Detection
Identifying types helps you spot "dirty" data quickly. If a column labeled "Age" suddenly contains the string "Unknown" or a negative number, knowing that the variable *should* be a positive integer allows you to set up automated validation rules to catch these outliers.

In [ ]:
df_numerical = df.select_dtypes(include='number').columns
df_object = df.select_dtypes(include=['object']).columns
df_discreet = df.select_dtypes(include=['category']).columns
df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
print(df_numerical)
print(df_object)
print(df_discreet)
print(df_categorical_features)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['target'], kde=True, color='blue')
plt.title('Distribution of Target Variable')
plt.xlabel('Target Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='categorical_1', data=df, order=df['categorical_1'].value_counts().index)
plt.title('Frequency of Categorical_1 Levels')
plt.xticks(rotation=45)
plt.show()

## Correlation


* **Pearson Correlation Coefficient ($\rho$):** Measures linear relationship strength.
    $$\rho_{X,Y} = \frac{\text{cov}(X,Y)}{\sigma_X \sigma_Y}$$
    Where values range from $-1$ (perfect negative) to $+1$ (perfect positive).


In [ ]:
# show correlation between the features
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# correlation matrix
sns.set(style="white")

# compute the correlation matrix
corr = df[df_numerical].corr().round(1)

# generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# set up the matplotlib figure
# f, ax = plt.subplots()
f = plt.figure(figsize=(12, 12))

# generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True);

plt.tight_layout()

In [ ]:
# calculate the correlation matrix
corr_matrix = df[df_numerical].corr()

# Create a mask for the upper triangle (to avoid duplicates)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Convert the correlation matrix to a long format
corr_df = corr_matrix.stack().reset_index()
corr_df.columns = ['feature1', 'feature2', 'correlation']

# Filter for correlations above a certain threshold (e.g., 0.7)
high_corr_df = corr_df[(abs(corr_df['correlation']) > 0.7) & (corr_df['feature1'] != corr_df['feature2'])]

# Sort by absolute correlation in descending order
high_corr_df = high_corr_df.sort_values(by='correlation', ascending=False, key=abs)

# Print the top correlated features
# print(high_corr_df['feature1'].to_list()[4:10])
print(high_corr_df)

# Create a variable to pickle
data = {'correlation scores': high_corr_df}

In [ ]:
# check for vif
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# handle null values (using mean imputation for simplicity)
x_copy = df.drop('target', axis=1)._get_numeric_data()
x_copy.fillna(x_copy.mean(), inplace=True)

print(max([variance_inflation_factor(x_copy, i) for i in range(x_copy.shape[1])]))

# calculate VIF
vif = pd.DataFrame()
vif["Variable"] = x_copy.columns
vif["VIF"] = [variance_inflation_factor(x_copy, i) for i in range(x_copy.shape[1])]
print(vif)

## Multicollinearity

* We want high correlation with target
* We don't want high correlation between features
* Drop correlated features
* Combine correlated features

* **Formula for Variance of $\hat{\beta}_j$:**
  $$\text{Var}(\hat{\beta}_j) = \frac{\sigma^2}{(n-1)\text{Var}(X_j)} \cdot \frac{1}{1-R_j^2}$$
* **The Impact:** As the correlation ($R_j^2$) between $X_j$ and other predictors approaches $1$, the variance of the estimate approaches infinity. This results in wide confidence intervals and high p-values, making it difficult to prove that a feature is statistically significant, even if it actually is.

We need to understand the relationships between our variables. In linear regression, we look for features that are highly correlated with our `target`, but *not* highly correlated with each other.

When independent variables are highly correlated with each other, it is called **multicollinearity**. Extreme multicollinearity can completely break a linear regression model.

Let's investigate our dataset to see if we have any issues.

#### 1. The Correlation Matrix
The correlation coefficient ($r$) measures the strength and direction of a linear relationship between two variables, ranging from -1 to 1. Let's visualize the $r$ values for every variable against every other variable using a heatmap.

#### 2. Investigating the Anomalies
**Stop and look at the heatmap. Answer the following:**
1.  Which feature has a moderate correlation (around 0.5 or -0.5) with our `target`?
2.  Do you see any variables that have a perfect correlation ($r = 1.0$) with each other (excluding the diagonal)?
3.  Do you see any variable that has a near-perfect correlation with our `target`?

*Hint: If a feature predicts the target perfectly, it might not be a real feature—it might be a "cheat code" or a proxy for the target itself. If two features are perfectly correlated, one of them is a clone!*

#### 3. The Mathematics of Redundancy: Variance Inflation Factor (VIF)
To formally test for multicollinearity, we use the Variance Inflation Factor (VIF). VIF measures how much the variance of an estimated regression coefficient is increased because of collinearity.

The formula relies on the $R^2$ value obtained by regressing that specific predictor against all *other* predictors:

$$VIF_i = \frac{1}{1 - R_i^2}$$

#### 4. The "Division by Zero" Mystery
If you got a `RuntimeWarning: divide by zero encountered in scalar divide` and VIF values of `inf` (infinity), congratulations! You just proved the math.

Look back at the VIF formula: $VIF_i = \frac{1}{1 - R_i^2}$.
If a feature can be *perfectly* predicted by the other features, its $R^2$ is exactly $1.0$.
Therefore, the denominator becomes $1 - 1 = 0$. Division by zero results in infinity!

**The Fix:** We must remove the duplicate clones and the "cheat code" variables before moving on to modeling.



In [ ]:
# Drop the target, the exact copies, and the 'perfect' linear combination
from statsmodels.tools.tools import add_constant

cols_to_drop = ['target', 'target_perfect', 'duplicate_1', 'duplicate_2']
cols_to_drop = [c for c in cols_to_drop if c in df.columns] # Safety check

X_clean = df.drop(cols_to_drop, axis=1).select_dtypes(include=['number'])
X_clean.fillna(X_clean.mean(), inplace=True)
X_clean = add_constant(X_clean)

# Recalculate VIF on the clean data
vif_clean = pd.DataFrame()
vif_clean["Feature"] = X_clean.columns
vif_clean["VIF"] = [variance_inflation_factor(X_clean.values, i) for i in range(X_clean.shape[1])]

print("Cleaned VIF Results:")
print(vif_clean)

*(A general rule of thumb is that VIF values > 5 or 10 indicate problematic multicollinearity. Notice how our cleaned dataset now has healthy VIF scores!)*

In [ ]:
# Iterative Feature Reduction (Removing High VIF)
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Start with a fresh copy of our clean data
x_copy1 = X_clean.copy()
removed_features = []
thresh = 10.0 # Standard threshold for severe multicollinearity

while True:
    # Calculate VIF for all current columns using .values to prevent warnings
    vif_values = [variance_inflation_factor(x_copy1.values, i) for i in range(x_copy1.shape[1])]

    # Create a Pandas Series for easier manipulation
    vif_series = pd.Series(vif_values, index=x_copy1.columns)

    # CRITICAL: We do not want to drop the 'const' (intercept) column!
    if 'const' in vif_series:
        vif_series = vif_series.drop('const')

    # If no features are left to check, break the loop
    if vif_series.empty:
        break

    # Find the highest VIF score among the actual features
    max_vif = vif_series.max()

    if max_vif > thresh:
        # Identify the name of the feature with the highest VIF
        max_index = vif_series.idxmax()
        print(f"Dropping '{max_index}' with VIF: {max_vif:.2f}")
        removed_features.append(max_index)

        # Drop the offending feature from our dataframe
        x_copy1 = x_copy1.drop(max_index, axis=1)
    else:
        # All remaining features have VIF below the threshold; we are done!
        break

# Calculate final VIF for the remaining features
vif_final = pd.DataFrame()
vif_final["Variable"] = x_copy1.columns
vif_final["VIF"] = [variance_inflation_factor(x_copy1.values, i) for i in range(x_copy1.shape[1])]

print("\nFinal Feature Set and their VIF scores:")
print(vif_final)

# Create a dictionary to pickle or pass to the next section
data = {'vif': vif_final, 'removed': removed_features}


## Outliers

* generate code that shows outliers

In [ ]:
# code along
df.boxplot(column=['outliers 1']);

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(x='pd qcut1', y='target', data=df, inner='quartile')
plt.title('Target Density by pd qcut1')
plt.show()

In [ ]:
# code along
df.describe()

In [ ]:
import pandas as pd

def count_outliers_iqr(df, column):
    """Counts the number of outliers in a DataFrame column using the IQR method."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers)

def detect_and_print_numerical_outliers_iqr(df):
    """
    Iterates through numerical columns in a DataFrame and prints the
    variable name with the number of outliers based on the IQR method.
    """
    numerical_cols = df.select_dtypes(include=['number']).columns
    for col in numerical_cols:
        num_outliers = count_outliers_iqr(df, col)
        print(f"Variable: {col}, Number of outliers (IQR): {num_outliers}")


detect_and_print_numerical_outliers_iqr(df[df_numerical])

## Percentiles and Box Plots

A **Percentile** is a measure used in statistics indicating the value below which a given percentage of observations in a group of observations falls.

* **Definition:** The $k^{th}$ percentile is the value such that $k\%$ of the data is less than or equal to that value.
* **Median ($Q_2$):** This is the $50^{th}$ percentile. Half the data is above it, and half is below.
* **Quartiles:** We commonly divide data into four equal parts:
    * **$Q_1$ (25th Percentile):** The "lower quartile."
    * **$Q_2$ (50th Percentile):** The Median.
    * **$Q_3$ (75th Percentile):** The "upper quartile."


| Feature | Statistical Value |
| :--- | :--- |
| **Lower Whisker Bound** | $Q_1 - 1.5 \times IQR$ |
| **Bottom of Box** | $25^{th}$ Percentile ($Q_1$) |
| **Middle Line** | $50^{th}$ Percentile ($Median$) |
| **Top of Box** | $75^{th}$ Percentile ($Q_3$) |
| **Upper Whisker Bound** | $Q_3 + 1.5 \times IQR$ |

## Sampling & Inference

Why the Mean Matters in Data Science

The sample mean is more than just an "average"; it is a foundational tool for statistical inference. Based on the principles of data science and statistical theory, here are the key talking points regarding its importance:

* **The Center of Gravity:** Mathematically and physically, the mean represents the balance point of a distribution. If you were to place a histogram on a pivot, the mean is the exact point where it would balance perfectly.
* **The "Smoother" Effect:** Calculating a mean can be viewed as an "equalizing" operation. It represents the value each individual in a collection would have if the total sum were redistributed evenly across all members.
* **Distribution Dependency:** The mean is determined solely by the distinct values in a collection and their relative proportions. This means any two datasets with the same underlying distribution will share the exact same mean, regardless of their size.
* **Proportions as Means:** In datasets with binary values ($0$ and $1$), the mean is mathematically equivalent to the proportion of $1$s. This is critical because it allows all mathematical properties and limit theorems of means to be applied directly to categorical proportions.
* **The Power of the Bell Shape:** A fundamental property of the sample mean is that its empirical distribution tends to become bell-shaped as the sample size increases, regardless of the shape of the population distribution.
* **A Universal Tool for Inference:** Because sample means behave predictably (forming a normal distribution) even when the population parameters are unknown, they are the primary tool used to make accurate inferences about large groups from small samples.
* **Unbiased Estimation:** The sample mean serves as an unbiased estimate of the population mean. As your sample size $n$ increases, the variability of this estimate decreases, leading to higher precision in your model.


## 10 Essential EDA Visualizations

### Prerequisites
```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your dataframe is named 'df'
# For demonstration, we handle the 'date' column conversion
df['date'] = pd.to_datetime(df['date'])
```

---

### 1. Histogram & KDE (Univariate Distribution)
Used to visualize the distribution of a single numerical variable, such as `target`. It helps identify skewness and kurtosis ($\gamma_2$).
```python
plt.figure(figsize=(10, 6))
sns.histplot(df['target'], kde=True, color='blue')
plt.title('Distribution of Target Variable')
plt.xlabel('Target Value')
plt.ylabel('Frequency')
plt.show()
```

### 2. Box Plot (Outliers and Quartiles)
Visualizes the five-number summary and identifies outliers based on the Interquartile Range ($IQR = Q_3 - Q_1$). Here, we look at `informative_1` across different `class` categories.
```python
plt.figure(figsize=(10, 6))
sns.boxplot(x='class', y='informative_1', data=df, palette='Set2')
plt.title('Informative_1 Distribution by Class')
plt.show()
```

### 3. Count Plot (Categorical Frequency)
The standard way to check for class imbalance in categorical variables like `categorical_1`.
```python
plt.figure(figsize=(10, 6))
sns.countplot(x='categorical_1', data=df, order=df['categorical_1'].value_counts().index)
plt.title('Frequency of Categorical_1 Levels')
plt.xticks(rotation=45)
plt.show()
```

### 4. Scatter Plot (Numerical Relationships)
Used to detect correlations or clusters between two continuous variables, such as `informative_1` and `target`.
```python
plt.figure(figsize=(10, 6))
sns.scatterplot(x='informative_1', y='target', hue='class', data=df, alpha=0.6)
plt.title('Informative_1 vs Target (Colored by Class)')
plt.show()
```

### 5. Correlation Heatmap (Multicollinearity)
A graphical representation of the correlation matrix where each cell represents the Pearson coefficient ($\rho$). Crucial for identifying redundant features.
```python
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Correlation Matrix Heatmap')
plt.show()
```

### 6. Pair Plot (Multivariate Overview)
Generates a grid of scatter plots and histograms for a subset of variables. Excellent for a quick overview of relationships.
```python
# Selecting a subset of relevant columns to avoid performance lag
cols = ['informative_1', 'informative_2', 'target', 'class']
sns.pairplot(df[cols], hue='class', diag_kind='kde')
plt.show()
```

### 7. Violin Plot (Density Estimation by Category)
Combines a box plot with a Kernel Density Estimate (KDE). It shows the peakiness and distribution of `target` across different `pd qcut1` categories.
```python
plt.figure(figsize=(10, 6))
sns.violinplot(x='pd qcut1', y='target', data=df, inner='quartile')
plt.title('Target Density by pd qcut1')
plt.show()
```

### 8. Bar Plot (Aggregated Statistics)
Unlike a count plot, this displays an aggregate statistic (like the mean $\mu$) of a numerical variable for each category.
```python
plt.figure(figsize=(10, 6))
sns.barplot(x='class', y='correlated w target 1', data=df, estimator='mean', errorbar='sd')
plt.title('Mean of Correlated_w_target_1 per Class (with Standard Deviation)')
plt.show()
```

### 9. Line Plot (Time Series Trends)
Since your data contains a `date` column, a line plot is essential to observe how the `target` variable changes over time $t$.
```python
plt.figure(figsize=(12, 6))
df_time = df.sort_values('date')
sns.lineplot(x='date', y='target', data=df_time)
plt.title('Target Variable Trend Over Time')
plt.xticks(rotation=45)
plt.show()
```

### 10. Joint Plot (Bivariate + Marginal Distributions)
Provides a scatter plot of two variables along with their individual histograms on the axes. Useful for seeing how $X$ and $Y$ relate while simultaneously seeing their individual spreads.
```python
sns.jointplot(x='multicollinearity 1', y='multicollinearity 2', data=df, kind='reg', color='purple')
plt.suptitle('Joint Plot of Multicollinear Features', y=1.02)
plt.show()
```


In [ ]:
df.to_csv('data science fiction ii pt 2.csv', index=False)

# Part 3 - Data Prep

https://www.udemy.com/course/feature-engineering-for-machine-learning

* Types and characteristics of data
* Missing data imputation
* Categorical encoding
* Variable transformation
* Discretization
* Outliers
* Datetime
* Scaling
* Feature creation

## Load Data

In [ ]:
import pandas as pd

df = pd.read_csv('data science fiction ii pt 2.csv')
print(df.shape)
print(df.info())
df.head()

## Clean the Data

In [ ]:
# constants

In [ ]:
# quasi constants

In [ ]:
# duplicate rows

In [ ]:
# duplicate features

In [ ]:
# missing data

In [ ]:
# scaling

In [ ]:
# outliers

## Identify Variable Types for Encoding

In [ ]:
# df_numerical = df.select_dtypes(include='number').columns
# df_object = df.select_dtypes(include=['object']).columns
# df_discreet = df.select_dtypes(include=['category']).columns
# df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
# print(df_numerical)
# print(df_object)
# print(df_discreet)
# print(df_categorical_features)

# Part 4 - Feature Engineering

Before we build our models to predict the `target` we need to make sure our data is in the right format. This is where **Feature Engineering** comes in.

Raw data from across the galaxy comes in wildly different scales. One sensor might measure atmospheric pressure in Pascals (ranging in the thousands), while another measures gravity in G-forces (ranging from 0 to 5). If we put these directly into a linear model, the math gets messy.

#### Standardization
To compare variables on equal footing, we convert them to **Standard Units** ($z$-scores). This transforms the data so that it has a mean of $0$ and a standard deviation ($SD$) of $1$.

The formula for standard units is:
$$z = \frac{x - \text{mean}(x)}{SD(x)}$$

Let's write a function to do this, applying the concepts from our Chapter 15 reading.

In [ ]:
# import numpy as np

# # 1. Create a temporary, unscaled feature for demonstration
# # Let's say Atmospheric Pressure on these planets averages 101,325 with a large variance
# df['temp_atmospheric_pressure'] = np.random.normal(loc=101325, scale=5000, size=len(df))

# def standard_units(any_numbers):
#     """
#     Converts an array of numbers to standard units (z-scores).
#     Subtracts the mean and divides by the standard deviation.
#     """
#     return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

# # 2. Apply standard units to our temporary feature
# df['temp_pressure_su'] = standard_units(df['temp_atmospheric_pressure'])

# # 3. Prove the math to the class
# print("--- BEFORE STANDARDIZATION ---")
# print(f"Mean of Pressure: {np.mean(df['temp_atmospheric_pressure']):.2f} Pascals")
# print(f"SD of Pressure:   {np.std(df['temp_atmospheric_pressure']):.2f}\n")

# print("--- AFTER STANDARDIZATION ---")
# # Using a small threshold to catch floating point rounding errors near zero
# print(f"Mean of Pressure (SU): {np.mean(df['temp_pressure_su']):.2f}")
# print(f"SD of Pressure (SU):   {np.std(df['temp_pressure_su']):.2f}\n")

# # 4. Clean up: Drop the temporary columns so they don't interfere with our actual modeling
# df.drop(['temp_atmospheric_pressure', 'temp_pressure_su'], axis=1, inplace=True)
# print("Temporary demonstration columns dropped.")

How does this fit with the Normal Curve and Percentiles, CDF, PPF?

In [ ]:
# # Calculate correlation (r) using the raw feature
# r_raw = np.corrcoef(df['informative_1'], df['target'])[0, 1]

# # Calculate correlation (r) using the standardized feature
# r_scaled = np.corrcoef(df['info_1_su'], df['target'])[0, 1]

# print(f"Correlation with raw data:      {r_raw:.4f}")
# print(f"Correlation with scaled data:   {r_scaled:.4f}")

It is a very common misconception that scaling your data makes a standard Linear Regression model predict the target more accurately. **The reality is that for Ordinary Least Squares (OLS) linear regression, scaling the features does not change the predictive power of the model at all.** The correlation coefficient ($r$), the $R^2$ value, and the final predictions will be exactly identical whether you use the raw data or the standardized data. The model simply adjusts the slope and intercept to compensate for the scale.

#### The Great Scaling Plot Twist
We just went through the trouble of standardizing our data. You might assume that this will make our linear regression model "better" at predicting the `target`. Let's test that hypothesis by checking the correlation coefficient ($r$) before and after scaling.

The numbers are exactly the same! Why?
Because standardization is a **linear transformation**. It shifts and stretches the data, but it does not change the fundamental pattern or relationship between the variables.

**So why do we bother scaling for Linear Regression?**
1. **Interpretability:** If `info_1` is measured in millions and `info_2` is measured in decimals, their regression coefficients (slopes) will look wildly different, making it hard to tell which feature is actually more important. Standardizing puts all coefficients on the same playing field.
2. **Future-Proofing:** While basic Linear Regression doesn't care about scale, advanced machine learning algorithms (like Neural Networks, K-Nearest Neighbors, and Penalized Regression) will completely fail if you don't scale your data first. It is a vital habit to build now.

For standard Ordinary Least Squares (OLS) linear regression, scaling the data does not improve the predictive accuracy of the model. The correlation coefficient ($r$), the $R^2$ value, and the final predictions will remain exactly identical whether you use raw or scaled data. The model simply adjusts the slope coefficients mathematically to compensate for the change in scale.

However, scaling remains a critical step for three main reasons:

1. **Coefficient Interpretability:** If one feature is measured in millions and another in tiny fractions, their resulting coefficients will be on completely different scales. Standardizing puts all features on a level playing field, allowing you to look at the coefficients and immediately tell which feature has the strongest impact on the target.
2. **Regularization (Ridge/Lasso):** If you move from OLS to penalized linear models, the algorithm applies a penalty to the size of the coefficients to prevent overfitting. If the data isn't scaled, features with naturally small numeric ranges will be unfairly penalized simply because of their units of measurement.
3. **Algorithmic Convergence & Stability:** Massive numbers mixed with tiny decimals can cause floating-point errors in computations. Furthermore, any algorithm that relies on distance metrics (like K-Nearest Neighbors) or gradient descent (like Neural Networks or Logistic Regression) requires scaled data to converge efficiently and correctly.

### Standardization (Z-Score Scaling)
Standardization transforms the data so that it has a mean ($\mu$) of $0$ and a standard deviation ($\sigma$) of $1$. It centers the data around zero and scales it based on the variance.

**The Math:**
$$z = \frac{x - \mu}{\sigma}$$

**Key Characteristics:**
* **No Bounding Box:** Standardization does not restrict the data to a specific range (like 0 to 1). If you have outliers, they will still exist as extreme values (e.g., a $z$-score of 4 or 5), but they won't compress the rest of your normal data into a tiny cluster.
* **Assumes Normal Distribution:** It is highly effective when the underlying raw data roughly follows a Gaussian (normal) distribution.
* **Best Used For:** Linear Regression, Logistic Regression, Support Vector Machines (SVMs), and Principal Component Analysis (PCA). These algorithms assume the data is centered around zero and that all features share a similar variance.

### Min-Max Scaling (Normalization)
Min-Max scaling violently compresses (or expands) all the data to fit within a strictly defined bounded interval—almost always between $0$ and $1$.

**The Math:**
$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

**Key Characteristics:**
* **Strict Boundaries:** Every single value will end up exactly between 0 and 1. The lowest value in the dataset becomes exactly $0.0$, and the highest becomes exactly $1.0$.
* **Highly Sensitive to Outliers:** Because the minimum and maximum values dictate the entire scale, a single massive outlier will crush all the normal data points into a tiny sliver of the $0$ to $1$ range.
* **Best Used For:** Neural Networks, image processing (where pixels naturally range from 0 to 255), and distance-based algorithms like K-Nearest Neighbors (KNN) when you absolutely need every feature to have the exact same weight and bounds.

A good rule of thumb: **Start with Standardization.** Because data in the real world (and in their science fiction dataset) is rarely perfectly bounded and frequently contains outliers, Standardization is generally more robust for the linear regression models you are teaching this week. Min-Max scaling is great, but only when they *know* the boundaries of their features or are moving into Deep Learning.

#### Variable Transformation & Derived Variables
Sometimes, the raw data doesn't capture the true relationship. We might need to transform a variable (like squaring it) or combine multiple variables to create a new, more predictive "derived" feature.

Let's imagine `info_1` represents "Solar Radiation" and `info_2` represents "Atmospheric Density". Perhaps the true predictor of our `target` (Resource Yield) isn't either of these alone, but how they interact!

Why do you think it might be dangerous to create *too many* derived variables in our dataset before running a linear regression? *(Hint: Think back to our discussion on VIF and multicollinearity!)*


In [ ]:
# # Create an interaction term (combining two features)
# # e.g., High radiation + high density might create a unique planetary condition
# df['radiation_density_interaction'] = df['info_1_su'] * df['info_2_su']

# # Create a polynomial transformation (squaring a feature)
# # e.g., Maybe resource yield increases exponentially with gravity, not linearly
# df['info_1_squared'] = df['informative_1'] ** 2

# # Let's peek at our newly engineered universe of features
# df[['informative_1', 'info_1_su', 'info_1_squared', 'radiation_density_interaction']].head()

Derived features retain the original information because they are built using deterministic mathematical rules applied directly to the raw data. The information isn’t lost; it is simply translated into a "language" that the machine learning model can understand more easily.

**The "Raw Ingredients" Analogy**

Think of your raw features like flour, sugar, and eggs. If you feed them to a judge (the model) separately, they might not taste like much. But if you mix the flour and sugar together and bake them (creating a derived feature: a cake), the original ingredients are absolutely still in there. You haven't lost the flour or the sugar; you have just combined them in a way that reveals their true value to the judge.

**Bending the Geometry (Non-Linear Transformations)**

Linear Regression is notoriously rigid—it can only draw straight lines or flat planes. If the true relationship in the universe is curved (for example, if a planet's Resource Yield grows exponentially as its Size increases), a straight line will be a terrible fit.

If we create a derived feature by squaring the planet's size ($Size^2$) or taking the logarithm ($log(Size)$), we are taking the exact same underlying information and physically bending the geometry of the scatter plot. We reshape the curve into a straight line so that our rigid linear model can finally slice through it perfectly. The information is retained, but its *shape* is changed.

**Capturing Context (Interaction Terms)**
Sometimes, information only matters in context.
Imagine two raw features: `Has_Atmosphere` (0 or 1) and `Water_Volume`.
* By itself, `Water_Volume` might not predict if a planet is habitable, because water in a vacuum just boils away.
* By itself, `Has_Atmosphere` might not predict habitability if the planet is bone dry.

By multiplying them together to create an **interaction term** (`Atmosphere_x_Water`), we retain the original information of both features but explicitly tell the model: *"Only pay attention to the water volume IF there is also an atmosphere."* We haven't added new data to our dataset; we have just connected the dots for the algorithm.

**A Caveat on "Loss"**
It is worth noting that some transformations *can* lose specific types of information. For example, if you derive a feature by taking the absolute value ($|x|$), you retain the *magnitude* of the original data but you permanently lose the *direction* (the positive/negative sign). As data scientists, we only make these derivations when we believe the lost information (the sign) is just noise, and the retained information (the magnitude) is the true signal.


# Part 5 - Feature Selection

In [ ]:
# # get data
# import pandas as pd

# df = pd.read_csv('data science fiction ii pt 3.csv')
# print(df.shape)
# print(df.info())
# df.head()

## Regularization (Lasso)

In the previous step, we manually dropped features by calculating VIF in a loop. While this works, modern machine learning provides a more elegant, mathematically rigorous way to handle multicollinearity and feature selection simultaneously: **Regularization**.

Regularization intentionally adds a "penalty" to the linear regression cost function. Instead of just finding the line with the smallest errors, the model must find the smallest errors *while using the smallest coefficients possible*.

There are two main types of regularization:
1.  **Ridge Regression (L2 Penalty):** Adds a penalty equal to the *square* of the magnitude of coefficients ($\lambda \sum \beta^2$). It shrinks coefficients of correlated features so they share the weight, preventing any single feature from dominating.
2.  **Lasso Regression (L1 Penalty):** Adds a penalty equal to the *absolute value* of the magnitude of coefficients ($\lambda \sum |\beta|$). **Lasso is a built-in feature selector.** If a feature is redundant or useless (like our `duplicate` columns), Lasso will shrink its coefficient to exactly $0.0$, completely removing it from the model!

**The Golden Rule of Regularization:** You **MUST** standardize your data before using Ridge or Lasso. Because the penalty is based on the numeric size of the coefficients, an unscaled feature measured in decimals will have a massive coefficient and get unfairly penalized compared to a feature measured in millions.

Let's use Lasso to let the algorithm find the "cheat codes" and duplicates for us!


In [ ]:
# from sklearn.linear_model import Lasso
# from sklearn.preprocessing import StandardScaler

# # 1. Grab all our numeric features (except the targets)
# # We will use the raw data before our VIF loop to see if Lasso can find the bad features itself
# X_all = df.drop(['target', 'target_perfect'], axis=1).select_dtypes(include=['number'])
# X_all.fillna(X_all.mean(), inplace=True)

# # 2. Standardize ALL features (The Golden Rule)
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_all)
# X_scaled_df = pd.DataFrame(X_scaled, columns=X_all.columns)

# # 3. Fit the Lasso Model
# # 'alpha' controls the strength of the penalty.
# # Higher alpha = more coefficients pushed to exactly 0.
# lasso = Lasso(alpha=5.0, random_state=42)
# lasso.fit(X_scaled_df, df['target'])

# # 4. View the Results
# lasso_results = pd.DataFrame({
#     'Feature': X_scaled_df.columns,
#     'Coefficient': np.round(lasso.coef_, 4)
# })

# # Separate the survivors from the discarded
# kept_features = lasso_results[lasso_results['Coefficient'] != 0]
# dropped_features = lasso_results[lasso_results['Coefficient'] == 0]

# print("--- FEATURES KEPT BY LASSO ---")
# print(kept_features.sort_values(by='Coefficient', ascending=False))

# print("\n--- FEATURES DROPPED TO 0.0 BY LASSO ---")
# print(dropped_features['Feature'].tolist())

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 1. Isolate the numeric features (excluding the target and our 'perfect' cheat column)
# numeric_features = df.drop(['target', 'target_perfect'], axis=1, errors='ignore').select_dtypes(include=['number'])

# # 2. Calculate correlations of all features against the target
# correlations = numeric_features.corrwith(df['target'])

# # 3. Find the top 5 STRONGEST correlations.
# # We use .abs() because a strong negative correlation (e.g., -0.8) is just as
# # predictive as a strong positive one!
# top_5_features = correlations.abs().sort_values(ascending=False).head(10).index

# # 4. Extract the actual correlation values (keeping their original +/- signs)
# top_5_corr_values = correlations[top_5_features]

# # 5. Plot the results using a horizontal bar chart
# plt.figure(figsize=(10, 6))
# # We use orient='h' for a horizontal bar plot, which makes feature names easier to read
# sns.barplot(x=top_5_corr_values.values, y=top_5_corr_values.index, palette='coolwarm')

# plt.title("Top 5 Features Most Highly Correlated with Target", fontsize=14)
# plt.xlabel("Correlation Coefficient (r)", fontsize=12)
# plt.ylabel("Feature Name", fontsize=12)

# # Add a vertical line at 0 so it's easy to see positive vs. negative correlations
# plt.axvline(x=0, color='black', linestyle='-', linewidth=1)
# plt.xlim(-1.1, 1.1) # Lock the x-axis to the mathematical bounds of r
# plt.grid(axis='x', linestyle='--', alpha=0.7)

# plt.show()

## Train Test Split

random_state was initialized in the first code cell

In [ ]:
# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis=1), df['class'], test_size=0.3, random_state=random_state)
# X_train.shape, X_test.shape

# Part 6 - Data Modeling and Evaluation

In Python, there are two dominant libraries for Linear Regression, and they serve two slightly different philosophies:
1. **Scikit-Learn (`sklearn`)**: Built for Machine Learning. It is highly optimized for building predictive pipelines and plugging into larger applications, but it doesn't give you much statistical background information.
2. **Statsmodels (`statsmodels.api`)**: Built for Statistical Inference. It provides a massive, detailed summary of your model's statistical health, including p-values and confidence intervals (which perfectly ties into our Chapter 16 reading!).

In [ ]:
# from sklearn.linear_model import LinearRegression

# # 1. Isolate our final, clean predictor variables (let's use the top 3 for simplicity)
# # Assuming 'top_5_features' was created in the previous step
# final_features = top_5_features[:3]
# X_final = df[final_features]

# # We must ensure there are no nulls before fitting
# X_final = X_final.fillna(X_final.mean())
# y = df['target']

# # 2. Initialize and fit the Scikit-Learn model
# sk_model = LinearRegression()
# sk_model.fit(X_final, y)

# # 3. Generate predictions (the y-hat values)
# df['predictions'] = sk_model.predict(X_final)

# print("Scikit-Learn Coefficients:")
# for feature, coef in zip(final_features, sk_model.coef_):
#     print(f"{feature}: {coef:.4f}")
# print(f"Intercept: {sk_model.intercept_:.4f}")

Scikit-Learn gave us the slope and intercept, but it didn't tell us if those numbers are statistically significant. What if our sample is just weird, and the true population slope is actually 0?

To answer that, we use `statsmodels`.

In [ ]:
# import statsmodels.api as sm

# # 1. Statsmodels requires us to explicitly add the intercept (constant)
# X_sm = sm.add_constant(X_final)

# # 2. Fit the Ordinary Least Squares (OLS) model
# # Note: statsmodels expects (Y, X) whereas sklearn expects (X, Y)
# ols_model = sm.OLS(y, X_sm).fit()

# # 3. Print the massive statistical summary
# print(ols_model.summary())

Look at the middle table of the `.summary()` output.
* **$P>|t|$ (P-value):** This tells us the probability that a feature's coefficient is actually 0 in the true population. If this is $> 0.05$, the feature might just be galactic noise!
* **$[0.025 \quad 0.975]$ (95% Confidence Interval):** If this interval crosses $0$ (e.g., ranges from $-2.5$ to $4.1$), we cannot confidently say this feature has a reliable impact on our target.

**Analyzing Residuals**

Chapter 16 explicitly warns us: *"All of the predictions and tests... assume that the regression model holds. Specifically... points that are on a straight line and then pushing them off the line by adding random normal noise."*

To verify this, we must plot the **Residuals** (the actual values minus our predicted values).
$$e_i = y_i - \hat{y}_i$$

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Calculate residuals
# df['residuals'] = df['target'] - df['predictions']

# # Plot Predictions vs. Residuals
# plt.figure(figsize=(10, 6))
# sns.scatterplot(x=df['predictions'], y=df['residuals'], alpha=0.6, edgecolor='k')

# # Draw the theoretical zero-error line
# plt.axhline(y=0, color='r', linestyle='--', linewidth=2)

# plt.title("Residual Plot: Checking for Non-Linearity", fontsize=14)
# plt.xlabel(r"Predicted Values ($\hat{y}$)", fontsize=12)
# plt.ylabel("Residuals (Error)", fontsize=12)
# plt.show()

If your residual plot looks like a completely random cloud of static around the red zero-line, congratulations! Your linear model is valid.

*However, if you see a distinct shape (like a U-curve, a funnel, or a wave)... what does that mean for your Science Fiction story in the final assignment?*